In [5]:
import mlflow
# setup to mlflow tracking server
mlflow.set_tracking_uri('http://13.51.166.199:5000')

In [6]:
# Set or create an experiment
mlflow.set_experiment("Exp. 4 - ML Algos with HP Tuning")

<Experiment: artifact_location='s3://mlfow-test60/7', creation_time=1776671282581, experiment_id='7', last_update_time=1776671282581, lifecycle_stage='active', name='Exp. 4 - ML Algos with HP Tuning', tags={}>

In [7]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import f1_score, accuracy_score
from imblearn.over_sampling import ADASYN
import mlflow
import mlflow.sklearn
import optuna
import seaborn as sns
import matplotlib.pyplot as plt

In [8]:
df = pd.read_csv('preprocessed_data.csv').dropna()
df.shape

(36662, 5)

# Vectorization And Resampling

In [9]:
# Define a function to vectorize the data using TF-IDF
def vectorize_data(X_train, X_val, X_test, max_features, ngram_range):
    vectorizer = TfidfVectorizer(max_features=max_features, ngram_range=ngram_range)
    X_train_vec = vectorizer.fit_transform(X_train['comment']).toarray()
    X_val_vec = vectorizer.transform(X_val['comment']).toarray()
    X_test_vec = vectorizer.transform(X_test['comment']).toarray()

    # Combine additional features
    X_train_combined = np.hstack([X_train_vec, X_train[['word_count', 'char_count', 'avg_word_length']].values])
    X_val_combined = np.hstack([X_val_vec, X_val[['word_count', 'char_count', 'avg_word_length']].values])
    X_test_combined = np.hstack([X_test_vec, X_test[['word_count', 'char_count', 'avg_word_length']].values])

    return X_train_combined, X_val_combined, X_test_combined

In [10]:
max_features = 1006
ngram_range = (1, 2)

# Split data into training, validation and testing sets
X = df[['comment', 'word_count', 'char_count', 'avg_word_length']]
y = df['category']

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.4, random_state=42, stratify=y_temp)

# Vectorize the data
X_train_combined, X_val_combined, X_test_combined = vectorize_data(X_train, X_val, X_test, max_features, ngram_range)

In [11]:
# Apply resampling technique
X_resampled, y_resampled = ADASYN(random_state=42).fit_resample(X_train_combined, y_train)

# Helper Function

In [12]:
# Define the function that evaluates the model on validation data
def evaluate_model(model, X_val, y_val):
    y_val_pred = model.predict(X_val)  # Predict on validation set
    f1 = f1_score(y_val, y_val_pred, average='macro')  # Calculate F1 (macro)
    accuracy = accuracy_score(y_val, y_val_pred)  # Calculate accuracy
    return f1, accuracy

# Random Forest

In [9]:
# Define the Optuna objective function
def objective(trial):
    # Hyperparameters to optimize
    n_estimators = trial.suggest_int("n_estimators", 50, 500, step = 10)
    max_depth = trial.suggest_int("max_depth", 5, 51, step = 2)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)
    max_features = trial.suggest_categorical("max_features", ["sqrt", "log2"])

    # Initialize the Random Forest model with the suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        random_state=42
    )

    # Fit the model on the resampled training data
    model.fit(X_resampled, y_resampled)

    # Evaluate the model on the validation set
    f1, accuracy = evaluate_model(model, X_val_combined, y_val)

    return accuracy, f1

In [10]:
# Run Optuna optimization
study = optuna.create_study(directions=["maximize", "maximize"], study_name = "Random_Forest_Optimization")  # Multi-objective optimization for both F1 and accuracy
study.optimize(objective, n_trials=50)

[I 2026-04-20 13:19:01,691] A new study created in memory with name: Random_Forest_Optimization
[I 2026-04-20 13:19:32,739] Trial 0 finished with values: [0.712675031823968, 0.6860526446593361] and parameters: {'n_estimators': 120, 'max_depth': 43, 'min_samples_split': 17, 'min_samples_leaf': 8, 'max_features': 'log2'}.
[I 2026-04-20 13:21:11,066] Trial 1 finished with values: [0.6826695762865975, 0.6505935792564754] and parameters: {'n_estimators': 220, 'max_depth': 17, 'min_samples_split': 8, 'min_samples_leaf': 6, 'max_features': 'sqrt'}.
[I 2026-04-20 13:26:55,709] Trial 2 finished with values: [0.7423167848699763, 0.7164596254061827] and parameters: {'n_estimators': 420, 'max_depth': 43, 'min_samples_split': 13, 'min_samples_leaf': 4, 'max_features': 'sqrt'}.
[I 2026-04-20 13:27:49,865] Trial 3 finished with values: [0.7412256773958902, 0.7155712342056008] and parameters: {'n_estimators': 60, 'max_depth': 49, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 'sqrt'}.

In [12]:
print(study.best_trials[:3])

[FrozenTrial(number=25, state=<TrialState.COMPLETE: 1>, values=[0.7512274959083469, 0.7256723744175445], datetime_start=datetime.datetime(2026, 4, 20, 14, 2, 45, 859118), datetime_complete=datetime.datetime(2026, 4, 20, 14, 4, 6, 687534), params={'n_estimators': 80, 'max_depth': 49, 'min_samples_split': 18, 'min_samples_leaf': 1, 'max_features': 'sqrt'}, user_attrs={}, system_attrs={'NSGAIISampler:generation': 0}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=50, step=10), 'max_depth': IntDistribution(high=51, log=False, low=5, step=2), 'min_samples_split': IntDistribution(high=20, log=False, low=2, step=1), 'min_samples_leaf': IntDistribution(high=10, log=False, low=1, step=1), 'max_features': CategoricalDistribution(choices=('sqrt', 'log2'))}, trial_id=25, value=None)]


In [ ]:
best_trial = sorted(study.best_trials, key=lambda t: t.values[0], reverse=True)[0]

with mlflow.start_run() as run:
    mlflow.set_tag("mlflow.runName", "Random Forest")
    mlflow.set_tag("resampling_technique", "Adasyn")
    mlflow.set_tag("vectorizer_type", "TF-IDF")

    # Log best trial parameters
    mlflow.log_params(best_trial.params)

    # Extract parameters from the best trial
    best_params = best_trial.params

    # Initialize the model using the best trial parameters with unpacking (**)
    model = RandomForestClassifier(random_state=42, **best_trial.params)

    # Train the model on the resampled training data
    model.fit(X_resampled, y_resampled)

    # Predictions on the test set
    y_test_pred = model.predict(X_test_combined)

    # Log classification metrics
    classification_rep = classification_report(y_test, y_test_pred, output_dict=True)
    accuracy = accuracy_score(y_test, y_test_pred)

    # Log accuracy
    mlflow.log_metric("accuracy", accuracy)

    # Log each metric from classification report
    for label, metrics in classification_rep.items():
        if isinstance(metrics, dict):
            for metric, value in metrics.items():
                mlflow.log_metric(f"{label}_{metric}", value)

    # Generate and log confusion matrix
    conf_matrix = confusion_matrix(y_test, y_test_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"Confusion Matrix - Random Forest")

    # Save and log the confusion matrix plot
    confusion_matrix_filename = "confusion_matrix.png"
    plt.savefig(confusion_matrix_filename)
    mlflow.log_artifact(confusion_matrix_filename)
    plt.close()

# Logistic Regression

In [15]:
from sklearn.linear_model import LogisticRegression

# Define the Optuna objective function for Logistic Regression
def objective_logistic_regression(trial):
    # Hyperparameters to optimize
    penalty = trial.suggest_categorical("penalty", ["l1", "l2"])
    C = trial.suggest_float("C", 1e-5, 1e5, log = True)  # Inverse of regularization strength

    # Initialize the Logistic Regression model with the suggested hyperparameters
    model = LogisticRegression(
        penalty=penalty,
        C=C,
        solver='liblinear',
        n_jobs=-1,
        max_iter=200,
        random_state=42)

    # Fit the model on the resampled training data
    model.fit(X_resampled, y_resampled)

    # Evaluate the model on the validation set
    f1, accuracy = evaluate_model(model, X_val_combined, y_val)

    return accuracy, f1

In [16]:
# Run Optuna optimization for Logistic Regression
study_logistic = optuna.create_study(directions=["maximize", "maximize"], study_name="Logistic_Regression_Optimization")  # Multi-objective optimization for both F1 and accuracy
study_logistic.optimize(objective_logistic_regression, n_trials=50,n_jobs=3)

[I 2026-04-20 15:41:49,311] A new study created in memory with name: Logistic_Regression_Optimization
C:\Users\adina\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1271: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 16.
  warnings.warn(
C:\Users\adina\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1271: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 16.
  warnings.warn(
[I 2026-04-20 15:41:50,501] Trial 0 finished with values: [0.7104928168757956, 0.6875063173001011] and parameters: {'penalty': 'l2', 'C': 0.026525670726299317}.
C:\Users\adina\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1271: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 16.
  warnings.warn(
C:\Users\adina\AppData\Loc

In [17]:
best_trial = sorted(study_logistic.best_trials, key=lambda t: t.values[0], reverse=True)[0]

with mlflow.start_run() as run:
    mlflow.set_tag("mlflow.runName", "Logistic Regression")
    mlflow.set_tag("resampling_technique", "Adasyn")
    mlflow.set_tag("vectorizer_type", "TF-IDF")

    # Log best trial parameters
    mlflow.log_params(best_trial.params)

    # Log algorithm name as a parameter
    mlflow.log_param("algo_name", "LogisticRegression")

    # Extract parameters from the best trial
    best_params = best_trial.params

    # Initialize the model using the best trial parameters with unpacking (**)
    model = LogisticRegression(solver='liblinear', random_state=42, **best_trial.params)

    # Train the model on the resampled training data
    model.fit(X_resampled, y_resampled)

    # Predictions on the test set
    y_test_pred = model.predict(X_test_combined)

    # Log classification metrics
    classification_rep = classification_report(y_test, y_test_pred, output_dict=True)
    accuracy = accuracy_score(y_test, y_test_pred)

    # Log accuracy
    mlflow.log_metric("accuracy", accuracy)

    # Log each metric from classification report
    for label, metrics in classification_rep.items():
        if isinstance(metrics, dict):
            for metric, value in metrics.items():
                mlflow.log_metric(f"{label}_{metric}", value)

    # Generate and log confusion matrix
    conf_matrix = confusion_matrix(y_test, y_test_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"Confusion Matrix - Logistic Regression")

    # Save and log the confusion matrix plot
    confusion_matrix_filename = "confusion_matrix.png"
    plt.savefig(confusion_matrix_filename)
    mlflow.log_artifact(confusion_matrix_filename)
    plt.close()

2026/04/20 15:58:42 INFO mlflow.tracking._tracking_service.client: 🏃 View run Logistic Regression at: http://13.51.166.199:5000/#/experiments/7/runs/703cc14ac63144ffb56f0f529930a22c.
2026/04/20 15:58:42 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://13.51.166.199:5000/#/experiments/7.


# Multinomial Naive Bayes

In [18]:
from sklearn.naive_bayes import MultinomialNB

# Define the Optuna objective function
def objective(trial):
    # Hyperparameters to optimize
    alpha = trial.suggest_float("alpha", 1e-4, 1.0)  # Laplace smoothing parameter
    fit_prior = trial.suggest_categorical("fit_prior", [True, False])  # Whether to learn class prior probabilities

    # Initialize the Naive Bayes model with the suggested hyperparameters
    model = MultinomialNB(
        alpha=alpha,
        fit_prior=fit_prior,
    )

    # Fit the model on the resampled training data
    model.fit(X_resampled, y_resampled)

    # Evaluate the model on the validation set
    f1, accuracy = evaluate_model(model, X_val_combined, y_val)

    return accuracy, f1

# Run Optuna optimization
study = optuna.create_study(directions=["maximize", "maximize"], study_name="Naive_Bayes_Optimization")
study.optimize(objective, n_trials=60,n_jobs=3)

[I 2026-04-20 16:04:32,174] A new study created in memory with name: Naive_Bayes_Optimization
[I 2026-04-20 16:04:32,407] Trial 1 finished with values: [0.5899254409892708, 0.5581011388308595] and parameters: {'alpha': 0.013072453610037576, 'fit_prior': False}.
[I 2026-04-20 16:04:32,422] Trial 0 finished with values: [0.5710129114384433, 0.5357785929199568] and parameters: {'alpha': 0.3276542845315345, 'fit_prior': False}.
[I 2026-04-20 16:04:32,429] Trial 2 finished with values: [0.5713766139298054, 0.5362984256867986] and parameters: {'alpha': 0.33713217770204307, 'fit_prior': True}.
[I 2026-04-20 16:04:32,632] Trial 3 finished with values: [0.5586470267321331, 0.5204143791697141] and parameters: {'alpha': 0.7848780684916725, 'fit_prior': True}.
[I 2026-04-20 16:04:32,637] Trial 4 finished with values: [0.5681032915075468, 0.5318249013560238] and parameters: {'alpha': 0.4551213209409747, 'fit_prior': True}.
[I 2026-04-20 16:04:32,645] Trial 5 finished with values: [0.575741043826150

In [19]:
best_trial = sorted(study.best_trials, key=lambda t: t.values[0], reverse=True)[0]

with mlflow.start_run() as run:
    mlflow.set_tag("mlflow.runName", "Mutlinomial Naive Bayes")
    mlflow.set_tag("resampling_technique", "Adasyn")
    mlflow.set_tag("vectorizer_type", "TF-IDF")

    # Log best trial parameters
    mlflow.log_params(best_trial.params)

    # Log algorithm name as a parameter
    mlflow.log_param("algo_name", "MutlinomialNB")

    # Extract parameters from the best trial
    best_params = best_trial.params

    # Initialize the model using the best trial parameters with unpacking (**)
    model = MultinomialNB(**best_trial.params)

    # Train the model on the resampled training data
    model.fit(X_resampled, y_resampled)

    # Predictions on the test set
    y_test_pred = model.predict(X_test_combined)

    # Log classification metrics
    classification_rep = classification_report(y_test, y_test_pred, output_dict=True)
    accuracy = accuracy_score(y_test, y_test_pred)

    # Log accuracy
    mlflow.log_metric("accuracy", accuracy)

    # Log each metric from classification report
    for label, metrics in classification_rep.items():
        if isinstance(metrics, dict):
            for metric, value in metrics.items():
                mlflow.log_metric(f"{label}_{metric}", value)

    # Generate and log confusion matrix
    conf_matrix = confusion_matrix(y_test, y_test_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"Confusion Matrix - Mutlinomial Naive Bayes")

    # Save and log the confusion matrix plot
    confusion_matrix_filename = "confusion_matrix.png"
    plt.savefig(confusion_matrix_filename)
    mlflow.log_artifact(confusion_matrix_filename)
    plt.close()

2026/04/20 16:05:30 INFO mlflow.tracking._tracking_service.client: 🏃 View run Mutlinomial Naive Bayes at: http://13.51.166.199:5000/#/experiments/7/runs/1c52e057e8764e859f92acfd6e58efa7.
2026/04/20 16:05:30 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://13.51.166.199:5000/#/experiments/7.


# SVM

In [1]:
from sklearn.svm import SVC

# Define the Optuna objective function
def objective(trial):
    # Hyperparameters to optimize
    C = trial.suggest_float("C", 1e-4, 1e2, log=True)  # Regularization strength
    kernel = trial.suggest_categorical("kernel", ["linear"])  # Kernel type

    # Initialize the SVM model with the suggested hyperparameters
    model = SVC(
        C=C,
        kernel=kernel,
        random_state=42
    )

    # Fit the model on the resampled training data
    model.fit(X_resampled, y_resampled)

    # Evaluate the model on the validation set
    f1, accuracy = evaluate_model(model, X_val_combined, y_val)

    return accuracy, f1

In [ ]:
# Run Optuna optimization
study_svm = optuna.create_study(directions=["maximize", "maximize"], study_name="SVM_Optimization")
study_svm.optimize(objective, n_trials=50, n_jobs=3)

[I 2026-04-20 16:18:27,645] A new study created in memory with name: SVM_Optimization
